# Klebsiella metadata

In [1]:
import pandas as pd

# No additional imports are needed; you already imported 'pandas as pd' above.
data_dir = "/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david"
metadata_file = data_dir + "/final/metadata_final_curated_slimmed.tsv"
output_dir = data_dir + "/processed/panaroo_with_reference_genome"

metadata = pd.read_csv(metadata_file, sep="\t", low_memory=False)

In [2]:
# Filter to those samples in kpsc_final_list True
kpsc_final_samples = metadata[metadata['kpsc_final_list']]
print(f"Number of samples in kpsc_final_list: {len(kpsc_final_samples)}")

Number of samples in kpsc_final_list: 79228


In [3]:
# Check how many kpsc_final_metadata have entries for assembly_file and gff_file
print(f"Number of kpsc_final with assembly_file: {kpsc_final_samples['assembly_file'].notna().sum()}")
print(f"Number of kpsc_final_metadata with gff_file: {kpsc_final_samples['gff_file'].notna().sum()}")

Number of kpsc_final with assembly_file: 79228
Number of kpsc_final_metadata with gff_file: 79228


In [4]:
# Double check that all with sublineage not null are in kpsc_final_list
sl_samples = kpsc_final_samples[kpsc_final_samples['Sublineage'].notna()]
# Number of samples with sublineage not null
print(f"Number of samples with sublineage not null: {len(sl_samples)}")
# Number of samples with sublineage not null and in kpsc_final_list
sl_samples = sl_samples[sl_samples['kpsc_final_list']]
print(f"Number of samples with sublineage not null and in kpsc_final_list: {len(sl_samples)}")

Number of samples with sublineage not null: 79225
Number of samples with sublineage not null and in kpsc_final_list: 79225


In [5]:
# Check we have one is_mgh78578 sample
global_reference_sample = kpsc_final_samples[kpsc_final_samples['is_mgh78578']]
print(f"Number of samples with is_mgh78578: {len(global_reference_sample)}")
# Print the sublineage and clonal group for the one sample
print(global_reference_sample[['Sublineage', 'Clonal group']])


Number of samples with is_mgh78578: 1
      Sublineage Clonal group
84821       SL38         CG38


In [6]:
# Ranking sublineages by size, choose all those with more than 100 samples
# For each, count how many are is_refseq True and how many is_complete_norway_genome True

strain_type_column = "Sublineage"
min_samples = 250

# Get value counts for each sublineage
strain_type_sizes = kpsc_final_samples[strain_type_column].value_counts()
filtered_strain_types = strain_type_sizes[strain_type_sizes > min_samples].index.tolist()

# For each of the largest sublineages, count the total and how many are is_refseq True
results = []
for sublineage in filtered_strain_types:
    total = strain_type_sizes[sublineage]
    n_is_refseq = kpsc_final_samples[
        (kpsc_final_samples[strain_type_column] == sublineage) & 
        (kpsc_final_samples['is_refseq'])
    ].shape[0]
    n_is_complete_norway_genome = kpsc_final_samples[
        (kpsc_final_samples[strain_type_column] == sublineage) & 
        (kpsc_final_samples['is_complete_norway_genome'])
    ].shape[0]
    # Take the species from the first sample in the sublineage
    species = kpsc_final_samples[
        (kpsc_final_samples[strain_type_column] == sublineage)
    ]['species'].iloc[0]
    results.append({
        'Sublineage': sublineage, 
        'count': total, 
        'count_is_refseq': n_is_refseq, 
        'count_is_complete_norway_genome': n_is_complete_norway_genome,
        'species': species
    })

sl_df = pd.DataFrame(results)
# Print the total number of samples in the sublineages with > min_samples samples
print(f"Total number of samples in the sublineages with > {min_samples} samples: {sl_df['count'].sum()}")
print(f"The total number of complete norway genomes in the sublineages with > {min_samples} samples: {sl_df['count_is_complete_norway_genome'].sum()}")
# print whole dataframe
display(sl_df)





Total number of samples in the sublineages with > 250 samples: 56240
The total number of complete norway genomes in the sublineages with > 250 samples: 199


,Sublineage,count,count_is_refseq,count_is_complete_norway_genome,species
0,SL258,16200,947,10,Klebsiella pneumoniae
1,SL147,5085,177,3,Klebsiella pneumoniae
2,SL17,4613,113,21,Klebsiella pneumoniae
3,SL307,4422,159,8,Klebsiella pneumoniae
4,SL15,3636,161,7,Klebsiella pneumoniae
5,SL14,2516,72,6,Klebsiella pneumoniae
6,SL37,1863,102,17,Klebsiella pneumoniae
7,SL45,1796,52,13,Klebsiella pneumoniae
8,SL101,1594,73,3,Klebsiella pneumoniae
9,SL231,1049,57,3,Klebsiella pneumoniae


In [7]:
# Pack sublineages (each < 500 samples) into greedy batches of >= 1500 total samples.
# We build batches from `final_kpsc_samples` only, and always save the final remainder batch.

max_sublineage_size = 250
# Sublineage sample counts (within the final kpsc set)
sl_sample_counts = kpsc_final_samples["Sublineage"].value_counts().sort_values(ascending=False)

# Only keep SLs below the max size
small_sl = sl_sample_counts[sl_sample_counts < max_sublineage_size]

# Metadata for the SLs below 500
small_sl_metadata = kpsc_final_samples[kpsc_final_samples['Sublineage'].isin(small_sl.index)]
# Look as 'species' counts for the SLs below 500
small_sl_species_counts = small_sl_metadata['species'].value_counts()
# Print the species counts for the SLs below 500
print(small_sl_species_counts)

species
Klebsiella pneumoniae                                 16002
Klebsiella variicola subsp. variicola                  3050
Klebsiella quasipneumoniae subsp. similipneumoniae     2496
Klebsiella quasipneumoniae subsp. quasipneumoniae      1080
Klebsiella quasivariicola                                76
Klebsiella africana                                      17
Klebsiella variicola subsp. tropica                      14
Name: count, dtype: int64


In [8]:
# # Slice out metadata for species == "Klebsiella pneumoniae"
# klebsiella_pneumoniae_metadata = small_sl_metadata[small_sl_metadata['species'] == "Klebsiella pneumoniae"]
# print(f"Number of samples in Klebsiella pneumoniae: {len(klebsiella_pneumoniae_metadata)}")

# # Slice metadata for each species in small_sl_metadata, save each to david/processed/panaroo_run/species_<species>.tsv
# for species in small_sl_species_counts.index:
#     # Skip Klebsiella pneumoniae - will batch this separately, by Sublineage
#     if species != "Klebsiella pneumoniae":
#         species_metadata = small_sl_metadata[small_sl_metadata['species'] == species]
#         # append the global reference sample to the species metadata
#         species_metadata = pd.concat([species_metadata, global_reference_sample])
#         print(f"Number of samples in {species}: {len(species_metadata)}")
#         species_metadata.to_csv(f"/home/dca36/rds/rds-floto-bacterial-4k08a2yyQLw/david/processed/panaroo_run/species_{species}.tsv", sep="\t")
#         print(f"Saved metadata for {species} to david/processed/panaroo_run/species_{species}.tsv")

In [9]:
# # For Klebsiella pneumoniae, batch by sublineage

# sl_list = klebsiella_pneumoniae_metadata['Sublineage'].unique().tolist()
# sl_sizes = klebsiella_pneumoniae_metadata['Sublineage'].value_counts()
# max_sublineage_size = 500
# print(f"Found {len(sl_list)} sublineages with < {max_sublineage_size} samples.")
# # Print top 10 sublineages
# print(sl_sizes.head(10))

# target_total = 1000
# batch_i = 0
# current_sls = []
# current_parts = []
# current_count = 0

# for sl in sl_list:
#     sl_df = final_kpsc_samples.loc[final_kpsc_samples["Sublineage"] == sl]

#     current_sls.append(sl)
#     current_parts.append(sl_df)
#     current_count += len(sl_df)

#     # Save when we reach/exceed the target (including the SL that pushes it over)
#     if current_count >= target_total:
#         batch_df = pd.concat(current_parts, ignore_index=True)
#         out_path = f"{out_dir}/kp_rare_sublineage_batch_{batch_i}.tsv"
#         batch_df.to_csv(out_path, sep="\t")

#         print(f"Saved batch {batch_i}: {current_count} samples across {len(current_sls)} sublineages -> {out_path}")

#         batch_i += 1
#         current_sls = []
#         current_parts = []
#         current_count = 0

# # Save final remainder batch (often < 1500)
# if current_parts:
#     batch_df = pd.concat(current_parts, ignore_index=True)
#     out_path = f"{out_dir}/next_sublineage_batch_{batch_i}.tsv"
#     batch_df.to_csv(out_path, sep="\t")
#     print(f"Saved final partial batch {batch_i}: {len(batch_df)} samples across {len(current_sls)} sublineages -> {out_path}")


In [10]:
# Build sublineage/K_locus combinations for shared K_locus values

# Keep only K_locus values that occur in more than one sublineage
shared_k_locus = (
    kpsc_final_samples.groupby('K_locus')['Sublineage']
    .nunique()
    .loc[lambda s: s > 1]
    .index
)

# Total size of each sublineage across all samples
sublineage_totals = (
    kpsc_final_samples.groupby('Sublineage')
    .size()
    .rename('SL_total')
)

# Number in each sublineage with each shared K_locus
sl_k_locus_counts = (
    kpsc_final_samples[kpsc_final_samples['K_locus'].isin(shared_k_locus)]
    .groupby(['Sublineage', 'K_locus'])
    .size()
    .rename('SL_with_K_locus')
    .reset_index()
)

# Per-row table (one row per Sublineage / K_locus)
matching_table = (
    sl_k_locus_counts
    .merge(sublineage_totals.reset_index(), on='Sublineage', how='left')
    .sort_values(['K_locus', 'SL_with_K_locus'], ascending=[True, False])
)

# One row per K_locus: parallel lists + total count
k_locus_summary = (
    matching_table.groupby('K_locus', as_index=False)
    .agg(
        Sublineages=('Sublineage', list),
        SL_with_K_locus=('SL_with_K_locus', list),
        SL_totals=('SL_total', list),
    )
)
k_locus_summary['total_with_K_locus'] = k_locus_summary['SL_with_K_locus'].apply(sum)
k_locus_summary = k_locus_summary.sort_values('total_with_K_locus', ascending=False)

display(k_locus_summary.head(10))

,K_locus,Sublineages,SL_with_K_locus,SL_totals,total_with_K_locus
137,KL64,"[SL147, SL258, SL14, SL395, SL17, SL505, SL30,...","[3415, 2195, 533, 262, 58, 57, 54, 50, 39, 39,...","[5085, 16200, 2516, 850, 4613, 57, 55, 1049, 6...",6947
8,KL107,"[SL258, SL307, SL17, SL1807, SL442, SL11887, S...","[5440, 17, 14, 12, 12, 11, 10, 7, 7, 6, 5, 3, ...","[16200, 4422, 4613, 34, 100, 11, 10, 8, 290, 6...",5596
3,KL102,"[SL307, SL268, SL147, SL152, SL442, SL10, SL17...","[4394, 186, 82, 79, 68, 56, 44, 32, 28, 21, 14...","[4422, 950, 5085, 421, 100, 66, 4613, 290, 28,...",5157
94,KL2,"[SL14, SL25, SL86, SL39, SL395, SL65, SL493, S...","[1692, 588, 390, 361, 331, 216, 120, 102, 65, ...","[2516, 588, 390, 885, 850, 216, 120, 102, 68, ...",4415
7,KL106,"[SL258, SL101, SL147, SL383, SL15, SL11988, SL...","[2947, 105, 30, 21, 14, 8, 6, 5, 4, 3, 2, 2, 2...","[16200, 1594, 5085, 294, 3636, 9, 6, 319, 219,...",3161
124,KL51,"[SL17, SL231, SL252, SL147, SL258, SL292, SL12...","[1543, 980, 99, 79, 63, 33, 32, 23, 18, 12, 11...","[4613, 1049, 392, 5085, 16200, 125, 32, 36, 18...",2989
99,KL24,"[SL15, SL258, SL45, SL17, SL1198, SL35, SL661,...","[963, 918, 736, 65, 17, 17, 15, 14, 14, 14, 13...","[3636, 16200, 1796, 4613, 54, 778, 407, 15, 50...",2896
1,KL10,"[SL15, SL147, SL551, SL3010, SL76, SL34, SL629...","[472, 457, 139, 87, 86, 61, 55, 50, 47, 34, 31...","[3636, 5085, 152, 501, 315, 530, 73, 78, 116, ...",1978
135,KL62,"[SL45, SL48, SL348, SL39, SL17, SL10077, SL15,...","[536, 400, 176, 128, 87, 51, 47, 43, 38, 34, 3...","[1796, 566, 201, 885, 4613, 51, 3636, 184, 38,...",1970
100,KL25,"[SL17, SL258, SL134, SL1552, SL607, SL792, SL4...","[1109, 115, 99, 78, 76, 61, 58, 52, 47, 31, 26...","[4613, 16200, 118, 78, 96, 61, 184, 217, 54, 3...",1960


In [11]:
# Look at dates pre 2000 and pre 1955 vs current top sublineages (top 10 by sample count)

_df = kpsc_final_samples.copy()
_year_col = "year_parsed"
if _year_col not in _df.columns:
    raise KeyError(f"Expected metadata column {_year_col!r} for year")

_df["_year"] = pd.to_numeric(_df[_year_col], errors="coerce")

top10_sl = _df["Sublineage"].value_counts().head(10)
top10_sl_names = top10_sl.index.tolist()

print("Top 10 sublineages in kpsc_final_samples (by sample count):")
display(top10_sl.to_frame("n_samples"))


def _report_date_cohort(year_mask, label):
    sub = _df.loc[year_mask]
    print(f"\n--- {label} ---")
    print(f"Samples (valid year in range): {len(sub)}")
    if sub.empty:
        return
    sl_in_cohort = sub["Sublineage"].dropna()
    top10_present = sorted(set(top10_sl_names) & set(sl_in_cohort.unique()))
    top10_absent = [s for s in top10_sl_names if s not in top10_present]
    print(f"Top-10 sublineages present: {top10_present if top10_present else 'none'}")
    print(f"Top-10 sublineages absent: {top10_absent}")
    counts = sl_in_cohort[sl_in_cohort.isin(top10_sl_names)].value_counts()
    if not counts.empty:
        print("Counts in cohort for those top-10 SL:")
        display(counts.to_frame("n_in_cohort"))


_n_missing_year = _df["_year"].isna().sum()
print(
    f"Samples with missing/non-numeric {_year_col}: {_n_missing_year} "
    f"(excluded from year-based cohorts below)"
)

_pre_2000 = _df["_year"].notna() & (_df["_year"] < 2000)
_pre_1955 = _df["_year"].notna() & (_df["_year"] < 1955)

_report_date_cohort(_pre_2000, "year < 2000")
_report_date_cohort(_pre_2000, "year < 1990")
_report_date_cohort(_pre_2000, "year < 1980")
_report_date_cohort(_pre_1955, "year < 1955")

Top 10 sublineages in kpsc_final_samples (by sample count):


,n_samples
Sublineage,
SL258,16200
SL147,5085
SL17,4613
SL307,4422
SL15,3636
SL14,2516
SL37,1863
SL45,1796
SL101,1594


Samples with missing/non-numeric year_parsed: 13687 (excluded from year-based cohorts below)

--- year < 2000 ---
Samples (valid year in range): 232
Top-10 sublineages present: ['SL14', 'SL147', 'SL15', 'SL231', 'SL258', 'SL307', 'SL37', 'SL45']
Top-10 sublineages absent: ['SL17', 'SL101']
Counts in cohort for those top-10 SL:


,n_in_cohort
Sublineage,
SL15,35
SL14,8
SL37,5
SL147,4
SL231,4
SL258,2
SL307,2
SL45,1



--- year < 1990 ---
Samples (valid year in range): 232
Top-10 sublineages present: ['SL14', 'SL147', 'SL15', 'SL231', 'SL258', 'SL307', 'SL37', 'SL45']
Top-10 sublineages absent: ['SL17', 'SL101']
Counts in cohort for those top-10 SL:


,n_in_cohort
Sublineage,
SL15,35
SL14,8
SL37,5
SL147,4
SL231,4
SL258,2
SL307,2
SL45,1



--- year < 1980 ---
Samples (valid year in range): 232
Top-10 sublineages present: ['SL14', 'SL147', 'SL15', 'SL231', 'SL258', 'SL307', 'SL37', 'SL45']
Top-10 sublineages absent: ['SL17', 'SL101']
Counts in cohort for those top-10 SL:


,n_in_cohort
Sublineage,
SL15,35
SL14,8
SL37,5
SL147,4
SL231,4
SL258,2
SL307,2
SL45,1



--- year < 1955 ---
Samples (valid year in range): 74
Top-10 sublineages present: ['SL37']
Top-10 sublineages absent: ['SL258', 'SL147', 'SL17', 'SL307', 'SL15', 'SL14', 'SL45', 'SL101', 'SL231']
Counts in cohort for those top-10 SL:


,n_in_cohort
Sublineage,
SL37,1


In [12]:
# Count total numebr of CGs in the dataset
print(f"Total number of CGs in the dataset: {len(kpsc_final_samples['Clonal group'].unique())}")

Total number of CGs in the dataset: 5552
